# L11 — Distributed serving: tensor parallel & autoscaling on Ray
**Objective 14.**

**Northfield Grocers context:** Saturday 10:00 traffic on the customer assistant is 4× Tuesday 15:00. Northfield needs the dense 31B model served TP=4, an autoscaler that pre-warms before known peaks and keeps two replicas for availability, and proof the service survives losing a worker mid-peak.

**Retail use cases:** Peak-hour scaling for the customer assistant; high availability for the store-ops assistant during trading hours.

**Platform:** Databricks multi-GPU (NC96ads_A100_v4, 4× A100). Steps 1–3 are NumPy; Step 4 needs the cluster.

**Done means:** sharded matmul equals full within 1e-6; autoscaler simulation keeps p95 queue time under the SLA; TP=4 endpoint survives a simulated worker loss.

## Step 1 — Tensor parallelism by hand
*Why:* TP splits a weight matrix across GPUs. Column-parallel: each GPU computes part of the output columns, then concatenate. Row-parallel: each GPU holds part of the input dimension, then all-reduce (sum). Do both and prove they equal the unsharded matmul.

In [1]:
# === Lab environment header (identical in every lab) ===
import os, sys, json, time, math, shutil, re, subprocess, importlib, importlib.util
import numpy as np, pandas as pd

def ensure_packages(pkgs):
    """Install any missing pip packages into THIS Python (same mechanism as %pip on Databricks) and import them.
    Fresh packages are importable immediately — no restart. Only labs that need extras call this (L02, L08)."""
    missing = [p for p in pkgs if importlib.util.find_spec(p.replace("-", "_")) is None]
    if not missing:
        print("Packages present:", pkgs); return
    print("Installing missing packages into", sys.executable, ":", missing)
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        r = subprocess.run(cmd + ["--break-system-packages"], capture_output=True, text=True)   # local system Pythons
    if r.returncode != 0:
        raise ImportError("pip could not install " + str(missing) + ". Ask the admin to add them as cluster libraries "
                          "(Compute → Libraries → PyPI) or use an internal index. pip said: " + r.stderr[-600:])
    importlib.invalidate_caches()
    for p in missing: importlib.import_module(p.replace("-", "_"))
    print("Installed and imported:", missing)

# Mode: "GPU" runs the full lab on Azure GPU compute; "SMOKE" runs the CPU/synthetic path anywhere.
LAB_MODE = os.environ.get("LAB_MODE") or ("GPU" if shutil.which("nvidia-smi") else "SMOKE")

# Data folder: env override → package-relative (../../data) → Unity Catalog volume → search the workspace once
_candidates = [os.environ.get("DATA_DIR"), os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data")), "/Volumes/northfield/llmops/labdata"]
DATA_DIR = next((c for c in _candidates if c and os.path.exists(os.path.join(c, "catalog_items.csv"))), None)
if DATA_DIR is None:
    import glob
    _hits = [h for root in ("/Workspace", "/Volumes", os.path.expanduser("~")) if os.path.isdir(root)
             for h in glob.glob(os.path.join(root, "**", "catalog_items.csv"), recursive=True)][:1]
    DATA_DIR = os.path.dirname(_hits[0]) if _hits else None
if DATA_DIR is None:
    raise FileNotFoundError("Lab data not found. Upload the package's data/ folder to a Unity Catalog volume and set "
                            "os.environ['DATA_DIR'] = '/Volumes/<catalog>/<schema>/<volume>' in a cell above this one.")
def gpu_only(msg):
    """Called wherever a step needs a GPU / model download that the smoke path cannot run."""
    print(f"[{LAB_MODE}] GPU-only step not executed here: {msg}")
def check(cond, msg):
    """Binary 'done means' assertion — prints PASS/FAIL and raises on FAIL so the notebook stops."""
    print(("PASS " if cond else "FAIL ") + msg); assert cond, msg
print(f"LAB_MODE={LAB_MODE}  DATA_DIR={DATA_DIR}  python={sys.version.split()[0]}")

LAB_MODE=SMOKE  DATA_DIR=/home/claude/llmops_labs/data  python=3.12.3


In [2]:
rng = np.random.default_rng(0); x = rng.normal(size=(8, 1024)).astype(np.float64); W = rng.normal(size=(1024, 4096)); TP = 4
full = x @ W

def column_parallel(x, W, tp):
    return np.concatenate([x @ Wi for Wi in np.array_split(W, tp, axis=1)], axis=1)

def row_parallel(x, W, tp):
    return sum(xi @ Wi for xi, Wi in zip(np.array_split(x, tp, axis=1), np.array_split(W, tp, axis=0)))

check(np.allclose(column_parallel(x, W, TP), full, atol=1e-6) and np.allclose(row_parallel(x, W, TP), full, atol=1e-6), "both shardings equal the full matmul")
print("bytes per GPU for W:", W.nbytes // TP, "vs full", W.nbytes)

PASS both shardings equal the full matmul
bytes per GPU for W: 8388608 vs full 33554432


## Step 2 — What TP costs: the all-reduce
*Why:* row-parallel needs a sum across GPUs every layer — that is why NVLink SKUs matter (L01). Count the communication volume per layer for the 8×4096 activation and compare it to the compute saved.

In [3]:
act_bytes = 8 * 4096 * 2   # bf16 activation
allreduce_bytes = 2 * (TP - 1) / TP * act_bytes   # ring all-reduce volume per GPU
print(f"all-reduce per layer per GPU ≈ {allreduce_bytes/1e3:.1f} KB; ×60 layers per token step")
check(allreduce_bytes > 0, "communication cost quantified")

all-reduce per layer per GPU ≈ 98.3 KB; ×60 layers per token step
PASS communication cost quantified


## Step 3 — Autoscaling simulation
*Why:* before touching a real autoscaler, simulate one. Arrivals follow a daily curve; each replica serves `cap` requests/s; scale up when queue p95 exceeds the SLA, scale down when utilisation is low, with a minimum of 2 replicas for HA. At minute 300 a replica dies.

In [4]:
def simulate(minutes=720, cap=20, sla_s=2.0, min_rep=2, max_rep=8, fail_at=300, scale_up_cooldown=5):
    rng = np.random.default_rng(0); replicas, queue, last_up = min_rep, 0.0, -99; log = []
    for t in range(minutes):
        rate = 25 + 60 * max(0, np.sin(np.pi * t / minutes)) + rng.normal(0, 3)
        if t == fail_at: replicas = max(min_rep - 1, replicas - 1)
        arrivals = 60 * rate; served = 60 * cap * replicas
        queue = max(0.0, queue + arrivals - served); wait = queue / (cap * replicas)
        if wait > sla_s * 0.8 and replicas < max_rep and t - last_up >= scale_up_cooldown: replicas += 1; last_up = t
        elif wait == 0 and arrivals < 0.5 * served and replicas > min_rep: replicas -= 1
        log.append(dict(t=t, rate=rate, replicas=replicas, wait_s=wait))
    return pd.DataFrame(log)

sim = simulate(); p95_wait = sim.wait_s.quantile(.95)
print(f"replicas min/max {sim.replicas.min()}/{sim.replicas.max()}; p95 queue wait {p95_wait:.2f}s; wait at failure+1 min {sim.wait_s[301]:.2f}s")
check(p95_wait < 2.0 and sim.replicas.min() >= 1, "autoscaler keeps p95 wait under SLA and recovers from replica loss")

replicas min/max 2/5; p95 queue wait 0.00s; wait at failure+1 min 0.00s
PASS autoscaler keeps p95 wait under SLA and recovers from replica loss


## Step 4 — Real TP=4 serving on Ray-on-Databricks (cluster)
*Why:* `ray.util.spark.setup_ray_cluster` starts Ray across Databricks workers; vLLM with `--tensor-parallel-size 4` shards Gemma 4 31B over the 4 A100s. Then kill one worker process and watch Ray Serve reschedule. Exact flags/APIs are version-sensitive — lab guide §Step 4.

In [5]:
if LAB_MODE == "GPU":
    gpu_only("Start Ray on Spark, launch vLLM TP=4, run the L03 harness at concurrency 32, simulate worker loss, record recovery time.")
recovery = {"tp4_throughput_tok_s": None, "recovery_s_after_worker_loss": None}
print(recovery); print("L11 complete." + ("" if LAB_MODE == "GPU" else " (SMOKE: Step 4 needs the multi-GPU cluster.)"))

{'tp4_throughput_tok_s': None, 'recovery_s_after_worker_loss': None}
L11 complete. (SMOKE: Step 4 needs the multi-GPU cluster.)
